In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
import pandas as pd

In [33]:
class MLP(nn.Module):
    def __init__(self, input_size):
        super(MLP, self).__init__()
        #структура нейросети: 19 - 128 - 64 - 32 - 1
        self.layers = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.layers(x)

In [34]:
mlp = MLP(input_size=19)
#функци потерь, сочетающая RMSE и MAE
criterion = nn.HuberLoss(delta=1.0)
optimizer = optim.Adam(mlp.parameters(), lr=0.001)

In [35]:
traffic = pd.read_csv('df_train_traffic_encoded_scaled.csv')

In [36]:
#делим на 60000, потому что остальные переменные в районе -1 и 1, веса становятся слишком большими и результаты становятся нестабильными
traffic['Трафик'] = traffic['Трафик'] / 60000
traffic

,Трафик,Численность населения,Количество домохозяйств,"Трафик пеший, в час","Трафик авто, в час",month_sin,month_cos,pca_1,pca_2,pca_3,pca_4,Населенный пункт_fold_traffic,Регион_fold_traffic,"Дата открытия, категориальный_Новый","Дата открытия, категориальный_Открыт давно","Дата открытия, категориальный_Средний по возрасту","Торговая площадь, категориальный_Большой","Торговая площадь, категориальный_Маленький","Торговая площадь, категориальный_Очень большой","Торговая площадь, категориальный_Средний"
0,0.994367,-0.182274,-0.533564,-0.543060,0.321416,-8.660254e-01,5.000000e-01,-0.500300,0.127879,0.066435,-1.215406,-0.245979,-0.121380,0,0,1,0,0,0,1
1,0.944567,-0.182274,-0.533564,-0.543060,0.321416,5.000000e-01,-8.660254e-01,-0.500300,0.127879,0.066435,-1.215406,-0.245979,-0.121380,0,0,1,0,0,0,1
2,0.858133,-0.182274,-0.533564,-0.543060,0.321416,5.000000e-01,8.660254e-01,-0.500300,0.127879,0.066435,-1.215406,-0.245979,-0.121380,0,0,1,0,0,0,1
3,0.944883,-0.182274,-0.533564,-0.543060,0.321416,1.224647e-16,-1.000000e+00,-0.500300,0.127879,0.066435,-1.215406,-0.245979,-0.121380,0,0,1,0,0,0,1
4,0.968800,-0.182274,-0.533564,-0.543060,0.321416,-5.000000e-01,-8.660254e-01,-0.500300,0.127879,0.066435,-1.215406,-0.245979,-0.121380,0,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
256718,0.861267,-0.188420,-0.608489,0.067017,0.549735,-8.660254e-01,5.000000e-01,-1.543277,0.099391,-0.088733,-0.136501,-1.117969,-0.880876,1,0,0,0,0,0,1
256719,0.858600,-0.188420,-0.608489,0.067017,0.549735,-5.000000e-01,8.660254e-01,-1.543277,0.099391,-0.088733,-0.136501,-1.117969,-0.880876,1,0,0,0,0,0,1
256720,0.826550,-0.188420,-0.608489,0.067017,0.549735,-1.000000e+00,-1.836970e-16,-1.543277,0.099391,-0.088733,-0.136501,-1.117969,-0.880876,1,0,0,0,0,0,1
256721,0.868583,-0.188420,-0.608489,0.067017,0.549735,-2.449294e-16,1.000000e+00,-1.543277,0.099391,-0.088733,-0.136501,-1.117969,-0.880876,1,0,0,0,0,0,1


In [37]:
y_traffic = traffic['Трафик']
x_traffic = traffic.drop(['Трафик'], axis=1)

In [38]:
from sklearn.model_selection import train_test_split

x_train_traffic, x_test_traffic, y_train_traffic, y_test_traffic = train_test_split(x_traffic, y_traffic, test_size=0.2, random_state=598)

In [39]:
#pytorch требует данных в формате тензоров
x_tensor = torch.tensor(x_train_traffic.values, dtype=torch.float32)
y_tensor = torch.tensor(y_train_traffic.values, dtype=torch.float32).reshape(-1, 1)

In [40]:
from torch.utils.data import DataLoader, TensorDataset
dataset = TensorDataset(x_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=100, shuffle=True)

In [41]:
from torch.nn.functional import l1_loss

In [ ]:
mlp.train()
for epoch in range(200):
    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        
        predictions = mlp(batch_x)
        
        loss = criterion(predictions, batch_y)
        
        loss.backward()
        optimizer.step()
    
    with torch.no_grad():
        full_predictions = mlp(x_tensor)
        rmse_train = torch.sqrt(
        criterion(full_predictions, y_tensor)) * 60000
        mae_train = l1_loss(full_predictions * 60000, y_tensor * 60000)
        
    print(
    f"Эпоха {epoch}: "
    f"RMSE {rmse_train.item():.2f} "
    f"MAE {mae_train.item():.2f}")

Эпоха 0: RMSE 6954.85 MAE 7406.99
Эпоха 1: RMSE 6824.35 MAE 7302.80
Эпоха 2: RMSE 7042.54 MAE 7465.11
Эпоха 3: RMSE 6727.49 MAE 7176.20
Эпоха 4: RMSE 6597.07 MAE 7079.83
Эпоха 5: RMSE 6537.15 MAE 7019.49
Эпоха 6: RMSE 6447.38 MAE 6892.44
Эпоха 7: RMSE 6367.94 MAE 6816.22
Эпоха 8: RMSE 6293.27 MAE 6794.63
Эпоха 9: RMSE 6147.83 MAE 6562.69
Эпоха 10: RMSE 6042.75 MAE 6502.14
Эпоха 11: RMSE 5946.92 MAE 6348.44
Эпоха 12: RMSE 5876.88 MAE 6301.22
Эпоха 13: RMSE 5872.44 MAE 6309.58
Эпоха 14: RMSE 5766.82 MAE 6136.27
Эпоха 15: RMSE 5641.66 MAE 6033.02
Эпоха 16: RMSE 5644.15 MAE 6029.42
Эпоха 17: RMSE 5551.08 MAE 5912.75
Эпоха 18: RMSE 5532.27 MAE 5905.17
Эпоха 19: RMSE 5470.75 MAE 5822.74
Эпоха 20: RMSE 5440.65 MAE 5815.52
Эпоха 21: RMSE 5382.13 MAE 5730.43
Эпоха 22: RMSE 5519.94 MAE 5863.77
Эпоха 23: RMSE 5376.86 MAE 5737.71
Эпоха 24: RMSE 5360.58 MAE 5692.67
Эпоха 25: RMSE 5315.83 MAE 5660.80
Эпоха 26: RMSE 5339.74 MAE 5683.27
Эпоха 27: RMSE 5242.62 MAE 5571.73
Эпоха 28: RMSE 5262.35 MAE 561

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
mlp.eval()
with torch.no_grad():
    x_test_tensor = torch.tensor(x_test_traffic.values, dtype=torch.float32)
    y_true_np = y_test_traffic.values.flatten() * 100000
    
    y_pred_tensor = mlp(x_test_tensor).flatten() * 100000
    y_pred_np = y_pred_tensor.numpy()
    
    mse_val = mean_squared_error(y_true_np, y_pred_np)
    mae_val = mean_absolute_error(y_true_np, y_pred_np)

print(f"MSE: {(mse_val**(1/2)):.2f}")
print(f"MAE: {mae_val:.2f} чел.")